# ProofLaunch Radeon submission gate

Run only after the kernel status is **Connected**. This Notebook does not publish, send, overwrite the original Gemma Notebook, or claim success before evidence exists.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys

print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', Path.cwd())


## 1. Unpack and verify the uploaded source


In [ ]:
archive = Path('prooflaunch-track1-submission-ready.tar.gz')
parts = sorted(Path('.').glob('prooflaunch-track1.part-*'))
if not archive.exists() and parts:
    with archive.open('wb') as destination:
        for part in parts:
            destination.write(part.read_bytes())
    print(f'reassembled {len(parts)} parts into {archive}')
if not archive.exists():
    archive = Path('prooflaunch-agent-remix-radeon-ready.tar.gz')
assert archive.exists(), f'Missing upload: {archive}'
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
print('archive:', archive)
print('sha256:', digest)
subprocess.run(['tar', '-xzf', str(archive)], check=True)


## 2. Run the deterministic control-plane gate


In [ ]:
repo = Path('prooflaunch_submission') if Path('prooflaunch_submission').exists() else Path('prooflaunch_agent_remix')
env = {**os.environ, 'PYTHONPATH': str(repo / 'src')}
tests = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', str(repo / 'tests'), '-v'],
    env=env, text=True, capture_output=True
)
print(tests.stdout)
print(tests.stderr)
assert tests.returncode == 0, 'Control-plane tests failed'


## 3. Record the AMD Radeon runtime


In [ ]:
import torch
runtime = {
    'torch_version': torch.__version__,
    'gpu_available': torch.cuda.is_available(),
    'gpu_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(runtime, indent=2))
assert runtime['gpu_available'], 'AMD Radeon GPU is not available'


## 4. Bind the fine-tuned Gemma classifier

This cell expects `predict_emotion_ft` to exist in the active kernel. If the original training Notebook uses a different kernel, load the saved LoRA adapter explicitly before continuing.


In [ ]:
from prooflaunch_remix.audience_pulse import GemmaAudiencePulse

assert 'predict_emotion_ft' in globals(), 'predict_emotion_ft is not loaded in this kernel'
audience_pulse = GemmaAudiencePulse(
    classifier=predict_emotion_ft,
    model_ref='google/gemma-4-E4B-it+emotion-lora:modelscope-single-radeon',
)
print('Gemma Audience Pulse adapter ready')


## 5. Required evidence before submission

- Record post-fine-tuning evaluation.
- Run one frozen real visual-generation fixture.
- Record model, seed, latency, peak VRAM, and output SHA-256.
- Complete audit → bounded repair → export.
- Record the 3–5 minute real-operation demo video.
- Change `SUBMISSION_GATE.json` to `ready` only when every P0 gate passes.
